In [2]:
import sys
sys.path.insert(0, '../')
import pandas as pd
import numpy as np
import time
from datetime import datetime
import matplotlib as mpl
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import os
import glob
import flats
from flats import Building
from flats import Flat
from flats import PV_system
import funcs
import copy

In [ ]:
# Manually added parameters

pv_scaler = 1 # kWh/h
price_scaler = 0.1 # €/MWh --> c/kWh
vat = 1.24 # Value added tax multiplicatior, e.g., vat = 1.24 means 24 % VAT
vat_deduction = 1.1/1.24 # VAT deduction for electricity, e.g., 1.1/1.24 means dudcting VAT from 24% to 10%
margin = 0.4 # Charged by electricity company, c/kWh
transfer_fee = 8.73 # Transfer fee and electricity tax, c/kWh
p_fixed = 7.86 # Fixed price, avg for 2-year contracts made in 2021, ref. https://energiavirasto.fi/sahkon-hintatilastot, read 2.7.2025
#p_fixed = 22.65 # 2022
p_ufn = [7.84, 7.86, 8.04, 8.57, 8.57, 8.03, 8.35, 8.49, 8.71, 9.02, 9.53, 9.78] # UFN-price for 2021 (the 2023 prices are rediculously high, so we use 2021 prices), c/kWh
#p_ufn = [38.23, 35.45, 30.24, 19.77, 20.92, 18.93, 17.03, 17.24, 17.05, 16.87, 18.00, 18.05] # UFN-price for 2023, c/kWh

prob_fixed = 0.45 # Probability of fixed price, 0 = spot price, 1 = fixed price, 0.5 = 50 % chance of either
prob_ufn = 0.24
flat_monitoring = True # If True, the flat monitoring is used
get_buildings = 'Read'
building_path = '../Files/Input_check/2025-06-27_Buildings_100.xlsx' # Path to the building file, if get_buildings is 'Read'

# Sami's initial notation
#roof_syst = ['MPV_12', 'MPV_45']
#facade_syst = ['MPV_facade','VBPV_large','VBPV_small','VBPV_hybrid'] # Facade PV systems


# Sami's automated notation
#roof_syst = ['2025-03-21_MPV_12_monofacial','2025-03-21_MPV_45_monofacial'] # Roof PV systems
#facade_syst = ['2025-03-21_MPV_Facade_monofacial','2025-03-21_VBPV_large_F_bifacial', '2025-03-21_VBPV_small_F_bifacial', '2025-03-21_VBPV_hybrid_F_bifacial'] # Facade PV systems
#roof_kWp = {'2025-03-21_MPV_12_monofacial': 5.28, '2025-03-21_MPV_45_monofacial': 19.8} # kWp for roof PV systems
#facade_kWp = {'2025-03-21_MPV_Facade_monofacial': 9.24, '2025-03-21_VBPV_large_F_bifacial': 9.24, '2025-03-21_VBPV_small_F_bifacial': 4.582, '2025-03-21_VBPV_hybrid_F_bifacial': 7.668} # kWp for facade PV systems

#print(roof_kWp['2025-03-21_MPV_12_monofacial']) # Print the kWp for the first roof system

# Magda's notation
#roof_syst = ['MPV_12_panels_35deg_monofacial', 'MPV_12_panels_south_monofacial', 'MPV_45_panels_35deg_monofacial', 'MPV_45_panels_south_monofacial']
#facade_syst = ['large_facade_panels_35deg_S_monofacial', 'large_facade_panels_south_S_monofacial', 'large_facade_panels_35deg_E_bifacial', 'large_facade_panels_south_E_bifacial', \
#               '158_small_panels_35deg_E_bifacial', '158_small_panels_south_E_bifacial', 'small_and_large_panels_35deg_E_bifacial', 'small_and_large_panels_south_E_bifacial']

#roof_kWp = {'MPV_12_panels_35deg_monofacial': 5.28, 'MPV_12_panels_south_monofacial': 5.28, 'MPV_45_panels_35deg_monofacial': 19.8, 'MPV_45_panels_south_monofacial': 19.8}
#facade_kWp = {'large_facade_panels_35deg_S_monofacial': 9.24, 'large_facade_panels_south_S_monofacial': 9.24, \
#             'large_facade_panels_35deg_E_bifacial': 9.24, 'large_facade_panels_south_E_bifacial': 9.24, \
#             '158_small_panels_35deg_E_bifacial': 4.582, '158_small_panels_south_E_bifacial': 4.582, \
#             'small_and_large_panels_35deg_E_bifacial': 7.668, 'small_and_large_panels_south_E_bifacial': 7.668}

energy_sharing = ['Simple', 'Disabled']#, 'Unconstrained'] # Energy sharing methods to be used. 'Simple' = simple energy sharing (no apartment-level analysis), 'Disabled' = no energy sharing, 'Unconstrained' = unconstrained energy sharing
naming_method = 'Magda'

# Magda's notation, the two systems needed for apartment-level analysis
roof_syst = ['MPV_12_panels_south_monofacial']
facade_syst = ['large_facade_panels_south_S_monofacial', 'large_facade_panels_south_E_bifacial']

roof_kWp = {'MPV_12_panels_south_monofacial': 5.28}
facade_kWp = {'large_facade_panels_south_S_monofacial': 9.24, \
             'large_facade_panels_south_E_bifacial': 9.24}

# File paths and variable names. Check these!
# NOTE: Old folders
#price_path = '../Input_files/Combined_Fingrid_Nordpool/DataCombined.xlsx' # Electricity price
#cons_path_start = './Files/2025-03-20_Initial_sims/Profiles_use/' # Consumption profiles to be used
#pv_prof_path = './Files/PV_profiles/2025-07-01_PV_shading/2025-07-01_Unshaded_ref.xlsx' # PV profiles. NOTE: The file must include profiles specified in roof_syst and facade_syst

# NOTE: Check folder
price_path = '../Files/Input_check/DataCombined.xlsx' # Electricity price
cons_path_start = '../Files/Input_check/Cons/' # Consumption profiles to be used
pv_prof_path = '../Files/Input_check/2026-01-20_Unshaded_check.xlsx' # PV profiles. NOTE: The file must include profiles specified in roof_syst and facade_syst

flat_path = '../Files/Input_check/2025-04-15_Flat_properties.xlsx' # Flat properties. Each row must represent a flat type and contain at least the file name of the electricity consumption
out_column_names = ['Total','SC','sur'] # Names for the output file columns

# NOTE: Remember to change these!
datenote = '2026-03-26' # Date to be added to the file names
identifier = '_check' # Additional info to file name
identifier_add = '_mix' # Additional info to file name
out_file_ext = '.xlsx' # File type

In [ ]:
# Run this section to rename files in the current folder
# NOTE: This is just to fix mistakes

folder = '.'  # Current folder
old_part = 'unshaded'  # Part of the filename to be replaced
new_part = 'good_shading'  # New part to replace the old part

for filename in os.listdir(folder):
    if filename.endswith('.xlsx') and old_part in filename:
        new_filename = filename.replace(old_part, new_part)
        os.rename(os.path.join(folder, filename), os.path.join(folder, new_filename))
        print(f"Renamed: {filename} -> {new_filename}")

In [ ]:
# Get the buildings

flat_info = pd.read_excel(flat_path,index_col=0) # Download the flat parameters
n_flats = np.array([2,8,4,2,2]) # Numbers of the flats in building, based on the flat type
n_buildings = 10 # Number of buildings to be created, if get_buildings is 'Create' or read from file, if get_buildings is 'Read'

# Create the buildings
if get_buildings == 'Create':
    out_file = datenote + '_Buildings_100' + out_file_ext
    buildings = funcs.create_buildings(n_buildings, flat_info, n_flats, cons_path_start, 'random', prob_fixed, prob_ufn, \
                                        True, out_file)
elif get_buildings == 'Read':
    sheets = pd.ExcelFile(building_path).sheet_names
    sheets.remove('Total_consumption') # Get the sheet names from the Excel file
    buildings = funcs.read_buildings(building_path, flat_info, n_flats, sheets[:n_buildings], cons_path_start)
    funcs.save_building_info(buildings, datenote + '_run_10_' + building_path.split('/')[-1])

else:
    print('Error: invalid value for get_buildings.')

print([buildings[i].area for i in range(len(buildings))]) # Print the areas of the buildings
print([buildings[i].get_total_consumption().sum() for i in range(len(buildings))]) # Print the total consumption of the building
print(sum(flat.area for flat in buildings[0].flats)) # Print the total area of the building



In [ ]:
# This section reads the input data

#pv_prod = pd.read_excel(pv_prof_path, index_col=0, sheet_name='Profiles') # Download PV data
pv_prod = pd.read_excel(pv_prof_path, index_col=0, sheet_name='Sheet1') # Download PV data

print(f"Annual PV productions as kWh\n{np.sum(pv_prod,axis=0)}") # Print the annual productions

price = pd.read_excel(price_path,sheet_name='Matlab',index_col=0)['Hinta']*price_scaler # Download the price, convert €/MWh -> c/kWh
vat_price = price.copy()
vat_price[price > 0] = price[price > 0] * vat # Implement VAT if price > 0
print(f"Mean price is {np.round(np.mean(price),2)} c/kWh.") # Print the annual mean (no VAT) price

# Create electricity sell and purchase profiles
vat_price_corr = vat_price.copy() # Create a copy of the VAT price for correction
vat_price_corr[(vat_price_corr > 0) & (vat_price.index.month < 5)] = vat_price_corr[(vat_price_corr > 0) & (vat_price.index.month < 5)]*vat_deduction # Deducted VAT from January to April

spot_no_corr = (vat_price + margin + transfer_fee) * 0.01
purchase_spot = (vat_price_corr + margin + transfer_fee) * 0.01 # Savings (not buying electricity), *0.01 converts c -> €

purchase_fixed_no_corr = pd.Series(index=purchase_spot.index, data=np.ones(len(purchase_spot))*p_fixed) # Create fixed_price series
purchase_fixed = purchase_fixed_no_corr.copy() # Create a copy of the fixed price for correction
purchase_fixed[purchase_fixed.index.month < 5] = purchase_fixed[purchase_fixed.index.month < 5]*vat_deduction # Deducted VAT from January to April

purchase_fixed_no_corr = (purchase_fixed_no_corr + transfer_fee) * 0.01 # Fixed price, *0.01 converts c -> €
purchase_fixed = (purchase_fixed + transfer_fee) * 0.01 # Fixed price, *0.01 converts c -> €

purchase_ufn = pd.Series(index=purchase_spot.index, data=np.ones(len(purchase_spot))) # Create empty series for unidirectional fixed price
purchase_ufn_no_corr = purchase_ufn.copy() # Create a copy

for m in range(12):
    if m < 4: # If month is January to April, use the VAT deduction
        price_ufn = p_ufn[m] * vat_deduction # Apply VAT deduction for UFN prices in Jan-Apr
    else:
        price_ufn = p_ufn[m]
    purchase_ufn[purchase_ufn.index.month == m+1] = purchase_ufn[purchase_ufn.index.month == m+1]*(price_ufn + transfer_fee) * 0.01 # Unidirectional fixed price, *0.01 converts c -> €
    purchase_ufn_no_corr[purchase_ufn_no_corr.index.month == m+1] = purchase_ufn_no_corr[purchase_ufn_no_corr.index.month == m+1]*(p_ufn[m] + transfer_fee) * 0.01 # Unidirectional fixed price, *0.01 converts c -> €

sell_price = (price.copy() - margin) * 0.01 # Selling price, *0.01 converts c -> €
intra_price = price.copy() * 0.01 # Create the intra-building trade price price

means = [np.mean(purchase_spot), np.mean(purchase_fixed), np.mean(purchase_ufn), np.mean(sell_price)] # Calculate the means of the prices
print(f"Means of the prices: {means}") # Print the means of the prices
df_prices = pd.DataFrame(index=purchase_spot.index, data={'Purchase spot (€/kWh)': purchase_spot,
                                                            'Purchase fixed (€/kWh)': purchase_fixed,
                                                            'Purchase ufn (€/kWh)': purchase_ufn,
                                                            'Sell price (€/kWh)': sell_price,
                                                            'Intra-building price (€/kWh)': intra_price}) # Create a DataFrame for the prices
#df_prices.to_excel(f'./Files/Prices_{datenote}{identifier_add}{identifier}.xlsx') # Save the prices to an Excel file

fig, ax = plt.subplots()
ax.plot(purchase_spot, label='Purchase spot (€/kWh)')   # Plot the purchase spot price
#ax.plot(spot_no_corr[4000:4100], label='Purchase spot no corr (€/kWh)') # Plot the purchase spot price without VAT correction
ax.plot(purchase_fixed, label='Purchase fixed (€/kWh)') # Plot the purchase fixed price
#ax.plot(purchase_fixed_no_corr, label='Purchase fixed no corr (€/kWh)') # Plot the purchase fixed price without VAT correction
ax.plot(purchase_ufn, label='Purchase ufn (€/kWh)') # Plot the purchase fixed price
#ax.plot(purchase_ufn_no_corr, label='Purchase ufn no corr (€/kWh)') # Plot the purchase fixed price without VAT correction
ax.plot(sell_price, label='Sell price (€/kWh)') # Plot the sell price
ax.set_ylim([-0.05, 0.4]) # Set y-axis limits
ax.legend()

In [10]:
# This section sets the flat electricity purchase type to same for all flats
contract_type = 'Spot' # Contract type for the flats, 'Spot' or 'Fixed'
identifier_add = '_spot' # Additional info to file name
for building in buildings:
    for flat in building.flats:
        flat.add_purchase_contract(contract_type) # Set the contract type for the flat

if contract_type == 'Default':
    identifier_add = '_mix' # Additional info to file name
    #path = './'
    for building in buildings:
        for flat in building.flats:
            #b_info = pd.read_excel(path + datenote + '_' + building_path.split('/')[-1], sheet_name=building.name, index_col=0) # Read the building info
            b_info = pd.read_excel('2026-01-20_run_10_2025-06-27_Buildings_100.xlsx', sheet_name=building.name, index_col=0) # Read the building info
            flat.add_purchase_contract(b_info.loc[flat.name, 'Default_contract']) # Set the contract
    

In [ ]:
# This section prints the contract type of each flat in each building
for building in buildings:
    for flat in building.flats:
        print(f"Flat {flat.name} in {building.name} has {flat.purchase_contract} contract.") # Print the contract type of the flat



In [ ]:
# Does the main economic run

# Go through the energy sharing scenarios
s_time = time.time() # Start the timer
cols = ['Production','P_SC','P_sur','Value','V_SC','V_sur','H_cons', 'Self-consumption (%)', 'Self-sufficiency (%)', 'kWp', 'Value / kWp (EUR)', 'Buy','Sell']

# Dictionary for getting the correct purchase contract
purchase_dict = {'fixed': purchase_fixed,
              'spot': purchase_spot,
              'ufn': purchase_ufn}

# NOTE: Set this false if you don't need flat monitoring
flat_monitoring = True # If True, the flat monitoring is used

for building in buildings:
    for flat in building.flats:
        flat.cost_no_pv = np.sum(flat.consumption * purchase_dict[flat.purchase_contract.lower()]) # Calculate the cost of the electricity without PV

for scenario in energy_sharing:
    first_building = True # Reset the first building flag for each scenario

    for building in buildings:
        cons_tot = building.get_total_consumption() # Get the total consumption of the building
        building.PV_systems = [] # Remove possibly existing PV systems
    
        # Initalize
        pv_sums = pd.DataFrame(columns=cols)
        out_file_name = datenote + identifier + '_' + str(scenario) + '_' + str(building.name) + identifier_add + out_file_ext
        mdata = pd.DataFrame({'Scenario': [scenario], 'Building': [building.name], 'Date': [datetime.now().strftime('%Y-%m-%d %H:%M:%S')]})
        mdata.to_excel(out_file_name, index=False, sheet_name='Metadata') # Save metadata to the output file

        # Go through the roof system options
        for r_system in roof_syst:
        #for r_system in [s for s in roof_syst if 'south' in s]:
            # Create PV_system object and add it to the Building
            r_name = r_system
            r_prod = pv_prod[r_system]
            roof_syst_act = PV_system(r_name,r_prod)
            building.add_PV(roof_syst_act)

            # Go through the facade system options
            for f_system in facade_syst:
                # Create PV_system object and add it to the Building
                f_name = f_system
                f_prod = pv_prod[f_system]
                facade_syst_act = PV_system(f_name,f_prod)
                building.add_PV(facade_syst_act)

                kWp = roof_kWp[r_name] + facade_kWp[f_name] # Calculate the total kWp of the system
                print(f"Total kWp of the system {r_name} + {f_name}: {kWp} kWp.") # Print the total kWp of the system

                system_name = funcs.revise_profile_name(r_name, f_name, naming_method) # Create a name for the current PV system

                # Check if the orientations match
                if system_name == 'Error: Orientations do not match!':
                    print(f"Error: Orientations do not match for {r_name} and {f_name}. Skipping this system.")
                    building.remove_PV(f_name) # Remove the facade system
                    continue # Continue to the next system

                print(f"Analyzing system {system_name} for scenario {scenario}.")


                # Get the total generation and create name for the current PV system
                pv_prod_tot = building.get_total_production()
                print(f"Total annual PV generation: {np.round(np.sum(pv_prod_tot))} kWh.")

                for flat in building.flats:
                    flat.production = pv_prod_tot*flat.area/building.area # Share of the PV system depending on the floor area

                # Case: no constraints in energy sharing, whole building one entity
                if scenario == 'Unconstrained':
                    
                    if flat_monitoring: # If flat monitoring is enabled, collect and save the flat monitoring data
                        building = funcs.add_flat_level_monitoring(building, ['pv_monitoring','value_monitoring']) # Add the production profiles to the flats in the building

                    building_shared = copy.deepcopy(building) # Create a deep copy of the building to share electricity
                    # Share the electricity between the flats in the building
                    # Note: This is a deep copy, so the original building remains unchanged
                    building_shared = funcs.share_electricity(building_shared, df_prices)
                    pv_prod_tot_share = building_shared.get_total_production() # Print the total production of the building after sharing
                    print(f"Total annual PV generation: {np.round(np.sum(pv_prod_tot_share))} kWh.")
                    if building_shared == building:
                        print("Objects have equal contents")
                    else:
                        print("Objects have different contents")
                
                    # Initialize the sums
                    p_sc = 0
                    p_sur = 0
                    v_sc = 0
                    v_sur = 0

                    for flat in building_shared.flats:
                        # Initialize
                        pv_coll, pv_coll_val, el_bill = funcs.calc_pv_value(flat.production, flat.consumption, purchase_fixed, purchase_ufn, purchase_spot, sell_price, out_column_names, flat.purchase_contract)
                        flat.el_bill = el_bill # Save the electricity bill with PV to the flat
                        # Collect annual sums
                        p_sc += np.sum(pv_coll['SC'])
                        p_sur += np.sum(pv_coll['sur'])
                        v_sc += np.sum(pv_coll_val['SC'])
                        v_sur += np.sum(pv_coll_val['sur'])

                    h_cons = np.sum(cons_tot)
                    buy = building_shared.buy
                    sell = building_shared.sell

                    # Add the annual sums to dataframe, index = PV system name
                    pv_sums.loc[system_name, :] = [p_sc+p_sur, p_sc, p_sur, v_sc+v_sur, v_sc, v_sur, h_cons, 100*p_sc/(p_sc+p_sur), 100*p_sc/h_cons, kWp, (v_sc+v_sur)/kWp, buy, sell]

                    if flat_monitoring: # If flat monitoring is enabled, save the flat monitoring data. NOTE: currently automatically enabled for Unconstrained scenario
                        df_flats = funcs.calculate_traded_electricity(building_shared)
                        for flat in building_shared.flats:
                            df_flats.loc[flat.name, 'El_bill'] = flat.el_bill # Add the cost of electricity without PV to the flat data
                            df_flats.loc[flat.name, 'El_cost_no_pv'] = flat.cost_no_pv # Add the cost of electricity without PV to the flat data
                            df_flats.loc[flat.name, 'Purchase_contract'] = flat.purchase_contract # Add the purchase contract to the flat data
                        with pd.ExcelWriter(out_file_name, mode='a') as writer:
                            df_flats.to_excel(writer, sheet_name=system_name + '_trade')

                # Case: no energy sharing, each apartment acts independently
                if scenario == 'Disabled':
                    # Initialize
                    #pv_coll = pd.DataFrame(index=price.index, columns=out_column_names, data=np.zeros([len(price.index), len(out_column_names)]))
                    #pv_coll_val = pv_coll.copy()

                    
                    if flat_monitoring: # If flat monitoring is enabled, collect and save the flat monitoring data
                        df_flats = pd.DataFrame(index=[flat.name for flat in building.flats]) # Add the production profiles to the flats in the building

                    # Initialize the sums
                    p_sc = 0
                    p_sur = 0
                    v_sc = 0
                    v_sur = 0
            
                    # Go through all flats in building
                    for flat in building.flats:
                        pv_coll, pv_coll_val, el_bill = funcs.calc_pv_value(flat.production, flat.consumption, purchase_fixed, purchase_ufn, purchase_spot, sell_price, out_column_names, flat.purchase_contract)
                        flat.el_bill = el_bill # Save the electricity bill with PV to the flat
                        p_sc += np.sum(pv_coll['SC'])
                        p_sur += np.sum(pv_coll['sur'])
                        v_sc += np.sum(pv_coll_val['SC'])
                        v_sur += np.sum(pv_coll_val['sur'])

                        if flat_monitoring:
                            df_flats.loc[flat.name, 'El_bill'] = flat.el_bill # Add the cost of electricity without PV to the flat data
                            df_flats.loc[flat.name, 'El_cost_no_pv'] = flat.cost_no_pv # Add the cost of electricity without PV to the flat data
                            df_flats.loc[flat.name, 'Purchase_contract'] = flat.purchase_contract # Add the purchase contract to the flat data

                    if flat_monitoring: # If flat monitoring is enabled, save the flat monitoring data
                        with pd.ExcelWriter(out_file_name, mode='a') as writer:
                            df_flats.to_excel(writer, sheet_name=system_name + '_flats')

                    # Add the annual sums to dataframe, index = PV system name
                    h_cons = np.sum(cons_tot)
                    pv_sums.loc[system_name,:] = [p_sc+p_sur, p_sc, p_sur, v_sc+v_sur, v_sc, v_sur, h_cons, 100*p_sc/(p_sc+p_sur), 100*p_sc/h_cons, kWp, (v_sc+v_sur)/kWp, 0, 0] # No buy or sell in this case

                if scenario == 'Simple':
                    prod = building.get_total_production() # Get the total production of the building
                    cons = building.get_total_consumption() # Get the total consumption of the building
                    pv_coll, pv_coll_val, el_bill = funcs.calc_pv_value(prod, cons, purchase_fixed, purchase_ufn, purchase_spot, sell_price, out_column_names, building.flats[0].purchase_contract)
                    h_cons = np.sum(cons_tot)
                    p_sc = np.sum(pv_coll['SC'])
                    p_sur = np.sum(pv_coll['sur'])
                    v_sc = np.sum(pv_coll_val['SC'])
                    v_sur = np.sum(pv_coll_val['sur'])
                    pv_sums.loc[system_name,:] = [p_sc+p_sur, p_sc, p_sur, v_sc+v_sur, v_sc, v_sur, h_cons, 100*p_sc/(p_sc+p_sur), 100*p_sc/h_cons, kWp, (v_sc+v_sur)/kWp, 0, 0] # No buy or sell in this case

                building.remove_PV(f_name) # Remove the facade system

            building.remove_PV(r_name) # Remove the rooftop system            
            print('Time elapsed: ' + str(np.round(time.time() - s_time, 2)) + ' seconds.\n')

        # Save the sums to file
        with pd.ExcelWriter(out_file_name, mode='a') as writer:
            pv_sums.to_excel(writer, sheet_name='Annual_sums')
        
        if first_building:
            pv_sums_avg = pv_sums.copy() # Copy the first building's sums to the average dataframe
            first_building = False
        else:
            pv_sums_avg += pv_sums.copy()

        print('Scenario ' + str(scenario) + ' completed for building ' + str(building.name))

    pv_sums_avg = pv_sums_avg / len(buildings) # Calculate the average sums
    out_name_avg = datenote + identifier + identifier_add + '_' + str(scenario) + '_avg' + out_file_ext
    pv_sums_avg.to_excel(out_name_avg, sheet_name='Annual_sums_avg') # Save the average sums to the output file
    